# PersonaLM: pretrain `s30m` on a Kaggle GPU

Public text only (English / Tagalog / Cebuano). Your private chats are NOT used here: fine-tune on your own laptop afterwards.

**Before running:** right side -> Session options -> Accelerator = **GPU T4 x2** (or P100). Add your dataset as Input (it must contain `persona_lm/`, `tok.json` and `data_pre/`).
**Run it with "Save Version" -> "Save & Run All (Commit)"**: it keeps running with the browser closed, and the checkpoint is kept as the notebook's Output.
**Continue later:** open the notebook, Add Input -> Notebook Output Files -> this notebook's previous version, then run again. It resumes by itself.

In [ ]:
# ---- settings (the only cell you may need to edit) ----
KAGGLE   = "/kaggle"      # (leave as is)
SIZE     = "s30m"
STEPS    = 18000          # 18000 steps x 16 x 2 x 512 = about 295M tokens = about 2 passes over your 147M tokens
BATCH    = 16             # micro-batch. If you get "out of memory", use 8 and ACCUM = 4
ACCUM    = 2              # BATCH x ACCUM must stay 32 so every step is the same size (16,384 tokens)
PRECISION = "auto"        # "auto" = fp16 on T4/P100. If the loss turns into nan, change to "fp32" (slower)
MAX_HOURS = 8.0           # safety stop: saves a checkpoint and ends cleanly before Kaggle's session limit

In [ ]:
import glob, os, shutil, subprocess, sys, time, json
W = os.path.join(KAGGLE, "working")
os.makedirs(W, exist_ok=True)

def first(pattern):
    hits = sorted(glob.glob(os.path.join(KAGGLE, "input", "**", pattern), recursive=True))
    return hits[0] if hits else None

pkg_init = first(os.path.join("persona_lm", "__init__.py"))
tok_path = first("tok.json")
data_pre = first(os.path.join("data_pre", "meta.json"))
assert pkg_init, "persona_lm folder not found in the Inputs: add your dataset"
assert tok_path, "tok.json not found in the Inputs"
assert data_pre, "data_pre folder (with meta.json) not found in the Inputs"
data_dir = os.path.dirname(data_pre)

PROJ = os.path.join(W, "proj")
shutil.rmtree(PROJ, ignore_errors=True)
os.makedirs(PROJ)
shutil.copytree(os.path.dirname(pkg_init), os.path.join(PROJ, "persona_lm"), ignore=shutil.ignore_patterns("__pycache__"))
RUN = os.path.join(W, "run_" + SIZE)
os.makedirs(RUN, exist_ok=True)

# resume: a checkpoint from an earlier version of this notebook (added as Input) is copied back
prev = [p for p in glob.glob(os.path.join(KAGGLE, "input", "**", "ckpt.npz"), recursive=True) if "run_" + SIZE in p]
if prev and not os.path.exists(os.path.join(RUN, "ckpt.npz")):
    prev.sort(key=os.path.getmtime)
    shutil.copy(prev[-1], os.path.join(RUN, "ckpt.npz"))
    print("resuming from", prev[-1])
else:
    print("checkpoint already in place" if os.path.exists(os.path.join(RUN, "ckpt.npz")) else "starting a NEW run")

shutil.copy(tok_path, os.path.join(W, "tok.json"))
print("package :", os.path.dirname(pkg_init))
print("data    :", data_dir)
print("run dir :", RUN)

In [ ]:
env = dict(os.environ, PYTHONPATH=PROJ, PYTHONUNBUFFERED="1")
def run(args, limit=None):
    cmd = [sys.executable, "-m", "persona_lm"] + args
    if limit:
        cmd = ["timeout", "-s", "INT", str(int(limit))] + cmd
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, cwd=PROJ, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

subprocess.run("nvidia-smi", shell=True)
run(["benchmark", "--size", SIZE, "--vocab", "16000", "--try-batches", "4,8,16", "--precision", PRECISION])

In [ ]:
# check that the GPU maths matches the reference (takes about a minute). Stop here if it says MISMATCH.
rc = run(["check-gpu"])
print("check-gpu exit code:", rc)

In [ ]:
# ---- training (hours). It saves a checkpoint every 10 minutes and again when it ends ----
t0 = time.time()
args = ["train", "--run", RUN, "--data-dir", data_dir, "--size", SIZE, "--ctx", "512",
        "--steps", str(STEPS), "--batch", str(BATCH), "--accum", str(ACCUM), "--precision", PRECISION]
rc = run(args, limit=MAX_HOURS * 3600)
print("exit code", rc, "after %.2f h" % ((time.time() - t0) / 3600))

In [ ]:
# ---- result ----
ck = os.path.join(RUN, "ckpt.npz")
sys.path.insert(0, PROJ)
from persona_lm.report import read_ckpt_meta
meta = read_ckpt_meta(ck) or {}
prevf = os.path.join(RUN, "ckpt.prev.npz")
if os.path.exists(prevf):
    os.remove(prevf)          # the older fallback copy only doubles the download size
print("checkpoint:", ck, "%.0f MB" % (os.path.getsize(ck) / 1e6))
print("step:", meta.get("step"), "of", STEPS)
if meta.get("step", 0) < STEPS:
    print("NOT finished yet: add this notebook's output as an Input in a new version and run again; it continues.")
else:
    print("DONE. Download run_%s/ckpt.npz from the Output tab (after the version finishes)." % SIZE)